In [1]:
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD
from scipy.stats import mannwhitneyu

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import statistics

# Hypervolume:

In [ ]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_names = ['wfg1']
objective_dim = 3
position_dim = 10

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2, 3]
choose_Xr_pool_type = [0,1,2] # [0, 1, 2]
choose_DE_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]

# Consider other results
insert_nsga2 = True
insert_old_mesh = True

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)
######################################################

In [ ]:
# List of file tuples
file_tuple_mesh = []
file_tuple_old_mesh = []
file_tuple_nsga2 = []

# Name of chosen files
file_tuple_mesh = [
        (f"MESH_G{i+1}S{j+1}D{k+1}_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}', exp_name)
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
        for exp_name in experiment_names
    ]

# Name of other files
if insert_old_mesh:
	file_tuple_old_mesh = [(f"OLD_MESH_G{i+1}S{j+1}D{k+1}_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1} (Original)', exp_name)
						for i in choose_global_best_attribution_type
						for j in choose_Xr_pool_type
						for k in choose_DE_mutation_type
						for exp_name in experiment_names
                        if os.path.exists(f"../results/OLD_MESH_G{i+1}S{j+1}D{k+1}_{exp_name}_{objective_dim}_{position_dim}.pkl")
					]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', exp_name) for exp_name in experiment_names]

file_tuple = file_tuple_mesh + file_tuple_old_mesh + file_tuple_nsga2

# Take the results
results_mesh = []
results_old_mesh = []
results_nsga2 = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_mesh.append(pickle.load(f))
for file_name, _, _ in file_tuple_old_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_old_mesh.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open("../results/" + file_name, "rb") as f:
		results_nsga2.append(pickle.load(f))
results = results_mesh + results_old_mesh + results_nsga2

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Median', 'Combined HV', 'Min. HV', 'Max. HV']

# Store the datas
data = [[
		alg_name,
		func_name.upper(),
		statistics.mean(HVs := [indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)]),
		statistics.stdev(HVs) if len(HVs) > 1 else 0,
		statistics.median(HVs) if len(HVs) > 1 else HVs[0],
		indicator(results[i]['combined'][1]),
		min(HVs),
		max(HVs),
		]
		for i, (_, alg_name, func_name) in enumerate(file_tuple)
		for r in [results[i]]
	]

# Create the DataFrame with hypervolume results
df = pd.DataFrame(data, columns=columns)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(df.sort_values(by=['Median'], ascending=False))

In [ ]:
print(df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

## Hypervolume Boxplot:

In [ ]:
# Experiment name
experiment_name = 'zdt4'
# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [0] # [0, 1, 2]
choose_DE_mutation_type = [0] # [0, 1, 2, 3, 4]
# Consider NSGA-II results
insert_nsga2 = False


# Name of chosen files
file_names_mesh = [
        f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
    ]
# Name of NSGA-II files
file_names_nsga2 = []
if insert_nsga2:
  file_names_nsga2.extend(['NSGA2_' + experiment_name + '.pkl'])

# Take the results
results_mesh = []
results_nsga2 = []
for i in range(len(file_names_mesh)):
  with open("../results/" + file_names_mesh[i], "rb") as f:
    results_mesh.append(pickle.load(f))
for i in range(len(file_names_nsga2)):
  with open("../results/" + file_names_nsga2[i], "rb") as f:
    results_nsga2.append(pickle.load(f))

# Get MESH hypervolumes
data = []
for i, fn in enumerate(file_names_mesh):
  HVs = []
  r = results_mesh[i]
  for j in range(len(results_mesh[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  data.append(HVs)
# Get NSGA-II hypervolumes if applicable
for i in range(len(file_names_nsga2)):
  HVs = []
  r = results_nsga2[i]
  for j in range(len(results_nsga2[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  data.append(HVs)

# Creating a boxplot
labels = ['MESH - ' + fn.split('_', 1)[0] for fn in file_names_mesh] + (['NSGA-II'] if insert_nsga2 else [])
plt.figure(figsize=(8, 5))
plt.boxplot(data, tick_labels=labels, showmeans=False)

# Titles and labels
# plt.title('Hypervolume - ' + experiment_name.upper(), fontsize=14)
plt.xlabel('Algoritmos') # plt.xlabel('Algorithms')
plt.ylabel('Hipervolume') # plt.ylabel('Hypervolume')
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.show()

# Inverse Generational Distance:

In [ ]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_names = ['wfg1']
objective_dim = 3
position_dim = 10

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2, 3]
choose_Xr_pool_type = [0,1,2] # [0, 1, 2]
choose_DE_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]

# Consider other results
insert_nsga2 = True
insert_old_mesh = True

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)
######################################################

# Teste de Hipóteses - Mann-Whitney U

In [ ]:
# Significance level
alpha = 0.05

# Store the groups for the Mann-Whitney U test
results_a = results_mesh
results_b = results_nsga2

# Create the pandas DataFrame columns
columns = ['Function', 'p-value', 'Statistic', 'H0', 'Median A', 'Median B', 'Mean A', 'Mean B', 'Std. Dev. A', 'Std. Dev. B']
# Store the datas
test_data = []
for i in range(min(len(results_a), len(results_b))):
  group_a = []
  group_b = []
  for j in range(len(results[i])-1):
    group_a.append(indicator(results_a[i][j+1]["F"]))
    group_b.append(indicator(results_b[i][j+1]["F"]))
  func = experiment_names[i]
  # Perform the Mann-Whitney U test
  stat, p_valor = mannwhitneyu(group_a, group_b, alternative='two-sided')
  test_data.append([func.upper(), p_valor, stat, 'Accepts' if p_valor > alpha else 'Rejects',
                    statistics.median(group_a), statistics.median(group_b),
                    statistics.mean(group_a), statistics.mean(group_b),
                    statistics.stdev(group_a), statistics.stdev(group_b)])

# Create the DataFrame for the Mann-Whitney U test results
df_statistic = pd.DataFrame(test_data, columns=columns)
df_statistic